<!-- paired-role-banner -->
> 🟩 **정답 코드 · 오른쪽 창** — 왼쪽과 같은 셀 위치의 참고 구현입니다. 필요한 한 줄만 확인하고 다시 실습 창으로 돌아가세요.

# 17. Embedding과 Vector Index — 직접 만들고 평가하기

외부 API와 다운로드 없이 `knowledge_base.jsonl`을
**문서 검증 → overlap chunking → sparse/dense embedding → 정규화 →
NumPy vector index → 저장/로드 → 검색 평가**까지 연결합니다.
작은 데이터이므로 RTX GPU가 없어도 실행되며,
GPU가 있으면 PyTorch dense encoder 학습에 자동으로 사용합니다.

In [ ]:
# 이 컴퓨터를 감지해 안전한 시작값을 확인합니다. 각 실습은 더 작은 값으로도 동작합니다.
from llm_engineering_lab.hardware import HardwareProfile, recommend_training_profile

LAB_HARDWARE = HardwareProfile.detect()
LAB_PROFILE = recommend_training_profile(LAB_HARDWARE)
print("hardware:", LAB_HARDWARE)
print("recommended profile:", LAB_PROFILE)

## 두 창 학습법

왼쪽에는 `notebooks/exercises/17_...ipynb`, 오른쪽에는 `notebooks/solutions/17_...ipynb`를 여세요.
먼저 왼쪽의 `TODO`를 직접 채우고 작은 입력으로 결과를 예상한 뒤 오른쪽 셀과 비교합니다.
정답을 복사하기보다 **입력 shape, 정규화 축, id/metadata 매핑**을 소리 내어 설명하는 것이 목표입니다.

## 전체 흐름

```text
JSONL → schema/id 검증 → overlap chunks
  ├─ char TF-IDF sparse vectors ─┐
  └─ tiny PyTorch dual encoder ──┼→ L2 normalize → cosine top-k
                                └→ metadata + vector 저장/로드 → Recall@k / MRR
```

실무의 FAISS/Qdrant/pgvector도 핵심 계약은 같습니다. `vector row i`가 어느 문서·chunk인지 잃지 않는 것이 특히 중요합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=random.seed,np.random.seed,torch.manual_seed,torch.cuda.manual_seed_all -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 4개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>random.seed(a=None)</code></summary>

#### `random.seed(a=None)`

- **역할:** Python 표준 난수 생성기의 상태를 초기화합니다.
- **입력·반환:** 정수 등의 시드를 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 같은 Python 난수 수열을 다시 만들어 실험을 재현합니다.
- **주의점:** NumPy와 PyTorch의 난수 상태는 별도로 고정해야 합니다.
- **공식 문서:** [random.seed](https://docs.python.org/3/library/random.html#random.seed)

</details>

<details>
<summary><code>np.random.seed(seed=None)</code></summary>

#### `np.random.seed(seed=None)`

- **역할:** NumPy의 기존 전역 난수 생성기 상태를 초기화합니다.
- **입력·반환:** 시드를 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 레거시 `np.random.*` 호출의 수열을 재현합니다.
- **주의점:** 새 코드에서는 독립 상태를 가진 `default_rng`가 권장됩니다.
- **공식 문서:** [np.random.seed](https://numpy.org/doc/stable/reference/random/generated/numpy.random.seed.html)

</details>

<details>
<summary><code>torch.manual_seed(seed)</code></summary>

#### `torch.manual_seed(seed)`

- **역할:** PyTorch 난수 생성기의 시드를 설정합니다.
- **입력·반환:** 정수 시드를 받고 `Generator`를 반환합니다.
- **이 셀에서 쓰는 이유:** 가중치 초기화와 텐서 샘플링을 반복 가능하게 만듭니다.
- **주의점:** 완전한 결정성에는 알고리즘·장치 설정도 추가로 필요할 수 있습니다.
- **공식 문서:** [torch.manual_seed](https://docs.pytorch.org/docs/stable/generated/torch.manual_seed.html)

</details>

<details>
<summary><code>torch.cuda.manual_seed_all(seed)</code></summary>

#### `torch.cuda.manual_seed_all(seed)`

- **역할:** 사용 가능한 모든 CUDA GPU의 난수 시드를 설정합니다.
- **입력·반환:** 정수 시드를 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 여러 GPU에서도 초기화와 샘플링을 최대한 반복 가능하게 합니다.
- **주의점:** CPU 시드와 결정론 설정을 대신하지 않습니다.
- **공식 문서:** [torch.cuda.manual_seed_all](https://docs.pytorch.org/docs/stable/generated/torch.cuda.manual_seed_all.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
import json, math, random, tempfile
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from llm_engineering_lab.acceleration import get_accelerator


def find_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "data" / "raw" / "knowledge_base.jsonl").exists():
            return candidate
    raise RuntimeError("코딩연습 폴더 또는 그 하위에서 실행하세요.")


ROOT = find_root()
KB_PATH = ROOT / "data" / "raw" / "knowledge_base.jsonl"
SEED = 17
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
print(ACCELERATOR.summary(), "| knowledge_base=", KB_PATH)

## 1) JSONL 로드와 데이터 계약

JSONL은 한 줄이 한 문서입니다. 필수 필드, 빈 문자열, 중복 id를 색인 전에 막아야 나중의 row/id 불일치를 피할 수 있습니다.
파일에는 UTF-8 BOM이 있을 수 있으므로 `utf-8-sig`로 읽습니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=json.loads -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>json.loads(s, ...)</code></summary>

#### `json.loads(s, ...)`

- **역할:** JSON 문자열을 Python 객체로 역직렬화합니다.
- **입력·반환:** 문자열·bytes를 받고 dict·list 등의 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** 메모리에 있는 JSON line이나 응답을 구조화합니다.
- **주의점:** 입력이 한 문서인지 JSON Lines의 한 줄인지 구분해야 합니다.
- **공식 문서:** [json.loads](https://docs.python.org/3/library/json.html#json.loads)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
REQUIRED_FIELDS = {"id", "title", "category", "content", "keywords"}


def load_and_validate_documents(path: Path) -> list[dict]:
    documents = []
    with path.open("r", encoding="utf-8-sig") as handle:
        for line_no, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            record = json.loads(line)
            missing = REQUIRED_FIELDS - record.keys()
            if missing:
                raise ValueError(f"line {line_no}: missing={sorted(missing)}")
            if any(not str(record[key]).strip() for key in ("id", "title", "content")):
                raise ValueError(f"line {line_no}: empty required value")
            if not isinstance(record["keywords"], list):
                raise TypeError(f"line {line_no}: keywords must be list")
            documents.append(record)
    ids = [document["id"] for document in documents]
    if len(ids) != len(set(ids)):
        raise ValueError("document id must be unique")
    return documents


documents = load_and_validate_documents(KB_PATH)
assert len(documents) == 12

<!-- api-explanations:start -->
<!-- api-explanations:keys=pd.DataFrame -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>pd.DataFrame(data=None, index=None, columns=None, ...)</code></summary>

#### `pd.DataFrame(data=None, index=None, columns=None, ...)`

- **역할:** 행·열 구조의 DataFrame을 만듭니다.
- **입력·반환:** 표 데이터와 열·그룹 설정을 받고 DataFrame, Series 또는 집계 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** 학습 전 데이터 분포·결측·중복과 분할 기준을 점검합니다.
- **주의점:** dtype 추론, 결측값, index 보존과 grouping 후 shape 변화를 확인해야 합니다.
- **공식 문서:** [pd.DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
document_frame = pd.DataFrame(
    {
        "id": d["id"],
        "title": d["title"],
        "category": d["category"],
        "content_chars": len(d["content"]),
        "keyword_count": len(d["keywords"]),
    }
    for d in documents
)
assert document_frame["id"].is_unique
display(document_frame)

## 2) overlap chunking과 metadata

긴 문서를 일정 길이로 자르면 query와 관련된 정보가 희석되지 않습니다. 경계에 걸린 문장을 보존하려고 이전 chunk 끝 일부를
다음 chunk에 겹칩니다. `chunk_id`, `source_id`, 문자 범위, 제목/카테고리를 함께 보존합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=dataclass -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>@dataclass(frozen=False, slots=False, ...)</code></summary>

#### `@dataclass(frozen=False, slots=False, ...)`

- **역할:** type annotation을 바탕으로 초기화·표현 메서드 등을 생성합니다.
- **입력·반환:** class decorator 설정을 받고 변환된 class를 반환합니다.
- **이 셀에서 쓰는 이유:** 실험 설정과 결과 record의 필드를 명시적으로 정의합니다.
- **주의점:** mutable 기본값은 `default_factory`를 쓰고 equality·frozen 의미를 확인합니다.
- **공식 문서:** [dataclass](https://docs.python.org/3/library/dataclasses.html#dataclasses.dataclass)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
@dataclass(frozen=True)
class Chunk:
    chunk_id: str
    source_id: str
    title: str
    category: str
    text: str
    start: int
    end: int
    keywords: tuple[str, ...]

    @property
    def searchable_text(self) -> str:
        return " ".join((self.title, self.category, *self.keywords, self.text))


def chunk_documents(
    records: list[dict], chunk_size: int = 120, overlap: int = 30
) -> list[Chunk]:
    if chunk_size <= 0 or not 0 <= overlap < chunk_size:
        raise ValueError("chunk_size > 0 and 0 <= overlap < chunk_size 이어야 합니다.")
    chunks = []
    for record in records:
        text = record["content"].strip()
        start, part = 0, 0
        while start < len(text):
            end = min(start + chunk_size, len(text))
            chunks.append(
                Chunk(
                    chunk_id=f"{record['id']}::c{part:03d}",
                    source_id=record["id"],
                    title=record["title"],
                    category=record["category"],
                    text=text[start:end],
                    start=start,
                    end=end,
                    keywords=tuple(record["keywords"]),
                )
            )
            if end == len(text):
                break
            start, part = end - overlap, part + 1
    return chunks

In [ ]:
chunks = chunk_documents(documents, chunk_size=120, overlap=30)
assert len({c.chunk_id for c in chunks}) == len(chunks)
assert all(0 <= c.start < c.end and c.end - c.start <= 120 for c in chunks)
assert {c.source_id for c in chunks} == {d["id"] for d in documents}
chunk_frame = pd.DataFrame(asdict(c) for c in chunks)
display(chunk_frame[["chunk_id", "source_id", "start", "end", "text"]].head(8))
print("documents=", len(documents), "chunks=", len(chunks))

## 3) sparse embedding: 문자 n-gram TF-IDF

한국어 형태소 분석기 없이도 문자 2~5 gram은 오류 코드, 합성어, 조사 변화에 비교적 강합니다. scikit-learn의 TF-IDF는 기본적으로
각 row를 L2 정규화하므로 내적이 cosine similarity가 됩니다. sparse matrix이므로 0을 저장하지 않습니다.

In [ ]:
tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True)
sparse_matrix = tfidf.fit_transform([chunk.searchable_text for chunk in chunks]).tocsr()
assert sparse_matrix.shape[0] == len(chunks)
sparsity = 1.0 - sparse_matrix.nnz / np.prod(sparse_matrix.shape)
print(
    "shape=",
    sparse_matrix.shape,
    "nnz=",
    sparse_matrix.nnz,
    "sparsity=",
    round(sparsity, 4),
)

<!-- api-explanations:start -->
<!-- api-explanations:keys=np.argsort -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>np.argsort(a, axis=-1, kind=None)</code></summary>

#### `np.argsort(a, axis=-1, kind=None)`

- **역할:** 정렬 결과를 만드는 index 순서를 반환합니다.
- **입력·반환:** 배열과 축·dtype 등의 설정을 받고 배열 또는 수치 결과를 반환합니다.
- **이 셀에서 쓰는 이유:** 데이터 shape과 수치 연산을 논문 실험에 맞게 구성·검증합니다.
- **주의점:** 축, dtype, 복사·view 여부와 빈 배열에서의 동작을 확인해야 합니다.
- **공식 문서:** [np.argsort](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
def sparse_search(query: str, top_k: int = 3) -> list[dict]:
    if not query.strip() or top_k <= 0:
        return []
    query_vector = tfidf.transform([query])
    scores = (sparse_matrix @ query_vector.T).toarray().ravel()
    order = np.argsort(scores)[::-1][:top_k]
    return [
        {
            "rank": rank,
            "score": float(scores[i]),
            "chunk_id": chunks[i].chunk_id,
            "source_id": chunks[i].source_id,
            "title": chunks[i].title,
        }
        for rank, i in enumerate(order, start=1)
    ]

In [ ]:
query = "카드에서 동일한 금액이 두 번 결제됐어요"
sparse_results = sparse_search(query, top_k=3)
assert "kb-003" in {row["source_id"] for row in sparse_results}
display(pd.DataFrame(sparse_results).round({"score": 3}))

## 4) dense embedding: 작은 PyTorch dual encoder

query encoder와 document encoder가 각각 문자열을 고정 길이 벡터로
바꿉니다. 여기서는 문자 embedding의 masked mean과 작은 projection을
사용합니다. 같은 batch의 다른 문서를 negative로 두는 in-batch
contrastive loss를 학습합니다. 12개 예시를 외우는 교육용 모델이며
범용 sentence embedding 모델이 아닙니다.

In [ ]:
QUERY_PAIRS = [
    ("비밀번호를 잊어버렸어요", "kb-001"),
    ("로그인 실패로 계정이 잠겼어요", "kb-002"),
    ("같은 주문이 두 번 결제됐어요", "kb-003"),
    ("구독 자동 갱신을 해지하고 싶어요", "kb-004"),
    ("환불이 카드에 언제 반영되나요", "kb-005"),
    ("운송장 배송 조회가 멈췄어요", "kb-006"),
    ("출고 뒤 배송지 주소를 바꿀 수 있나요", "kb-007"),
    ("파손된 상품을 교환하고 싶어요", "kb-008"),
    ("API 호출에서 401 인증 오류가 납니다", "kb-009"),
    ("429 rate limit 재시도 방법", "kb-010"),
    ("서버 5xx 장애를 확인하는 절차", "kb-011"),
    ("개인정보 내보내기와 계정 삭제", "kb-012"),
]
document_by_id = {d["id"]: d for d in documents}
train_document_texts = [
    " ".join(
        (
            document_by_id[i]["title"],
            *document_by_id[i]["keywords"],
            document_by_id[i]["content"],
        )
    )
    for _, i in QUERY_PAIRS
]
all_texts = (
    [q for q, _ in QUERY_PAIRS]
    + train_document_texts
    + [c.searchable_text for c in chunks]
)
characters = sorted(set("".join(all_texts)))
char_to_id = {char: index + 2 for index, char in enumerate(characters)}
char_to_id["<PAD>"] = 0
char_to_id["<UNK>"] = 1
print("vocabulary=", len(char_to_id))

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.zeros,torch.tensor,torch.bool -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 3개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.zeros(*size, dtype=None, device=None)</code></summary>

#### `torch.zeros(*size, dtype=None, device=None)`

- **역할:** 모든 원소가 0인 텐서를 만듭니다.
- **입력·반환:** shape·dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 초기 상태·mask·accumulator를 명시적으로 준비합니다.
- **주의점:** dtype과 device를 생략하면 주변 텐서와 달라질 수 있습니다.
- **공식 문서:** [torch.zeros](https://docs.pytorch.org/docs/stable/generated/torch.zeros.html)

</details>

<details>
<summary><code>torch.tensor(data, dtype=None, device=None)</code></summary>

#### `torch.tensor(data, dtype=None, device=None)`

- **역할:** 데이터를 복사해 새 PyTorch 텐서를 만듭니다.
- **입력·반환:** 배열형 데이터와 선택적 dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 상수·label·축소 실험 입력을 명시적으로 구성합니다.
- **주의점:** 기존 텐서에서 호출하면 복사가 생기므로 경우에 따라 `as_tensor`가 낫습니다.
- **공식 문서:** [torch.tensor](https://docs.pytorch.org/docs/stable/generated/torch.tensor.html)

</details>

<details>
<summary><code>torch.bool</code></summary>

#### `torch.bool`

- **역할:** 불리언 dtype
- **입력·반환:** 값이 아니라 텐서 원소의 저장 형식으로 사용됩니다.
- **이 셀에서 쓰는 이유:** mask와 조건 결과를 표현합니다.
- **주의점:** 연산마다 허용 dtype이 다르고 변환 시 정밀도나 메모리가 달라집니다.
- **공식 문서:** [torch.bool](https://docs.pytorch.org/docs/stable/tensor_attributes.html#torch.dtype)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
def batch_char_ids(
    texts: list[str], max_length: int = 180
) -> tuple[torch.Tensor, torch.Tensor]:
    rows = [[char_to_id.get(ch, 1) for ch in text[:max_length]] for text in texts]
    width = max(1, max(len(row) for row in rows))
    ids = torch.zeros((len(rows), width), dtype=torch.long)
    for row_index, row in enumerate(rows):
        ids[row_index, : len(row)] = torch.tensor(row, dtype=torch.long)
    return ids, ids.ne(0)


train_queries = [q for q, _ in QUERY_PAIRS]
query_ids, query_mask = batch_char_ids(train_queries)
document_ids, document_mask = batch_char_ids(train_document_texts)
assert query_ids.ndim == 2 and query_mask.dtype == torch.bool

<!-- api-explanations:start -->
<!-- api-explanations:keys=nn.Embedding,nn.Linear,Tensor.sum,F.normalize,ACCELERATOR.move -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 5개 · 수식 4개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>nn.Embedding(num_embeddings, embedding_dim, ...)</code></summary>

#### `nn.Embedding(num_embeddings, embedding_dim, ...)`

- **역할:** 정수 ID를 학습 가능한 벡터로 조회합니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
e_i=E[i],\qquad E\in\mathbb{R}^{V\times D}
$$

- **기호:** $i$는 정수 ID, $V$는 vocabulary 크기, $D$는 embedding 차원입니다.
- **수식 → 코드:** forward는 one-hot 행렬곱을 만들지 않고 weight 행 $E[i]$를 직접 조회합니다.
- **직관:** 범주 ID를 학습 가능한 연속 벡터 좌표로 바꿉니다.
- **shape 확인:** 입력 `[...] -> 출력 [..., D]`; 입력 dtype은 보통 `torch.int64`입니다.
- **공식 문서:** [nn.Embedding](https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html)

</details>

<details>
<summary><code>nn.Linear(in_features, out_features, bias=True)</code></summary>

#### `nn.Linear(in_features, out_features, bias=True)`

- **역할:** 완전연결 affine 변환을 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
y=xW^{\mathsf T}+b
$$

- **기호:** $x$는 입력, $W$는 weight, $b$는 bias, $y$는 출력입니다.
- **수식 → 코드:** `in_features`와 `out_features`가 각각 $W$의 두 축을 결정합니다.
- **직관:** 각 출력 feature는 모든 입력 feature의 학습 가능한 가중합입니다.
- **shape 확인:** `[..., D_in] -> [..., D_out]`, weight는 `[D_out, D_in]`입니다.
- **공식 문서:** [nn.Linear](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html)

</details>

<details>
<summary><code>tensor.sum(dim=None, keepdim=False, dtype=None)</code></summary>

#### `tensor.sum(dim=None, keepdim=False, dtype=None)`

- **역할:** 지정 축의 원소 합을 계산합니다.
- **입력·반환:** 텐서와 축 설정을 받고 축약된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 확률 질량, mask 개수, message aggregation을 계산합니다.
- **주의점:** 정수 overflow와 축약 뒤 shape 변화를 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
S=\sum_{i=1}^{N}x_i
$$

- **기호:** $N$은 선택 축의 원소 수입니다.
- **수식 → 코드:** `dim` 또는 `axis`의 모든 원소를 더해 축약합니다.
- **직관:** count, 확률 질량, 이웃 message처럼 누적량을 계산합니다.
- **shape 확인:** 선택한 축은 제거되거나 `keepdim=True`이면 크기 1로 남습니다.
- **공식 문서:** [Tensor.sum](https://docs.pytorch.org/docs/stable/generated/torch.sum.html)

</details>

<details>
<summary><code>F.normalize(input, p=2.0, dim=1, eps=1e-12)</code></summary>

#### `F.normalize(input, p=2.0, dim=1, eps=1e-12)`

- **역할:** 지정 축의 벡터 norm을 1로 정규화합니다.
- **입력·반환:** 예측·target과 설정을 받고 축약된 loss 또는 같은 shape의 loss를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 objective를 수치화해 역전파 시작점으로 사용합니다.
- **주의점:** 입력 형식, target dtype·shape, `reduction` 의미를 반드시 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\hat x=\frac{x}{\max(\lVert x\rVert_p,\epsilon)}
$$

- **기호:** $p$는 norm 차수, $\epsilon$은 0으로 나누지 않기 위한 하한입니다.
- **수식 → 코드:** 코드의 `dim` 축에서 norm을 구한 뒤 모든 원소를 같은 값으로 나눕니다.
- **직관:** 벡터 크기를 제거해 방향이나 상대 패턴에 집중하게 합니다.
- **shape 확인:** 입력과 출력 shape은 같고 선택 축의 norm은 약 1입니다.
- **공식 문서:** [F.normalize](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.normalize.html)

</details>

<details>
<summary><code>ACCELERATOR.move(value)</code></summary>

#### `ACCELERATOR.move(value)`

- **역할:** 텐서나 모듈을 선택된 실행 장치로 옮기는 프로젝트 도우미입니다.
- **입력·반환:** 이동할 값을 받고 같은 논리 값을 장치에 배치해 반환합니다.
- **이 셀에서 쓰는 이유:** 모델과 batch가 서로 다른 장치에 놓이는 오류를 막습니다.
- **주의점:** 컨테이너 재귀 처리 여부와 dtype 변경 여부는 프로젝트 구현을 확인해야 합니다.
- **공식 문서:** 프로젝트 내부 도우미이므로 정의의 docstring을 확인합니다.

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
class TinyDualEncoder(torch.nn.Module):
    def __init__(self, vocab_size: int, hidden: int = 64, output_dim: int = 48):
        super().__init__()
        self.query_embedding = torch.nn.Embedding(vocab_size, hidden, padding_idx=0)
        self.document_embedding = torch.nn.Embedding(vocab_size, hidden, padding_idx=0)
        self.query_projection = torch.nn.Linear(hidden, output_dim)
        self.document_projection = torch.nn.Linear(hidden, output_dim)

    def encode(self, ids: torch.Tensor, mask: torch.Tensor, tower: str) -> torch.Tensor:
        embedding = (
            self.query_embedding if tower == "query" else self.document_embedding
        )
        projection = (
            self.query_projection if tower == "query" else self.document_projection
        )
        token_vectors = embedding(ids)
        weights = mask.unsqueeze(-1).to(token_vectors.dtype)
        pooled = (token_vectors * weights).sum(1) / weights.sum(1).clamp_min(1.0)
        return torch.nn.functional.normalize(projection(pooled), p=2, dim=-1)


model = ACCELERATOR.move(TinyDualEncoder(len(char_to_id)))

## 5) contrastive training

정답 행렬의 대각선이 positive입니다.
`query_vectors @ document_vectors.T / temperature`는 `[B, B]` logits이고,
target은 `[0, 1, ..., B-1]`입니다. 작은 temperature는 점수 차이를 확대합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.optim.AdamW,Module.parameters,torch.arange,Module.train,Optimizer.zero_grad,F.cross_entropy,Tensor.backward,Optimizer.step,Tensor.detach -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 9개 · 수식 4개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.optim.AdamW(params, lr=1e-3, weight_decay=1e-2, ...)</code></summary>

#### `torch.optim.AdamW(params, lr=1e-3, weight_decay=1e-2, ...)`

- **역할:** weight decay를 gradient update와 분리한 AdamW optimizer를 만듭니다.
- **입력·반환:** 파라미터와 hyperparameter를 받고 optimizer 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** Transformer 계열에서 흔한 최적화 규칙을 재현합니다.
- **주의점:** bias·정규화 파라미터를 decay에서 제외하는 설정도 자주 사용합니다.

##### 관련 수식과 코드 연결

$$
\theta_t=(1-\eta\lambda)\theta_{t-1}-\eta\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}
$$

- **기호:** $\lambda$는 weight decay이고 나머지 기호는 Adam과 같습니다.
- **수식 → 코드:** 코드의 `weight_decay`가 gradient 기반 Adam update와 분리된 축소 항으로 적용됩니다.
- **직관:** 손실 gradient와 무관하게 큰 weight를 조금씩 줄이는 것이 Adam과의 핵심 차이입니다.
- **shape 확인:** 파라미터와 모멘트 tensor의 shape은 변하지 않습니다.
- **공식 문서:** [torch.optim.AdamW](https://docs.pytorch.org/docs/stable/generated/torch.optim.AdamW.html)

</details>

<details>
<summary><code>model.parameters(recurse=True)</code></summary>

#### `model.parameters(recurse=True)`

- **역할:** 모듈에 등록된 학습 파라미터 iterator를 반환합니다.
- **입력·반환:** 하위 모듈 포함 여부를 받고 `Parameter` iterator를 반환합니다.
- **이 셀에서 쓰는 이유:** optimizer가 갱신할 weight와 bias를 전달합니다.
- **주의점:** iterator는 한 번 소비되며 buffer와 일반 tensor attribute는 포함하지 않습니다.
- **공식 문서:** [Module.parameters](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>torch.arange(start=0, end, step=1, ...)</code></summary>

#### `torch.arange(start=0, end, step=1, ...)`

- **역할:** 일정 간격의 1차원 텐서를 만듭니다.
- **입력·반환:** 범위와 간격을 받고 1차원 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** position·index·time step을 벡터화해 만듭니다.
- **주의점:** 부동소수점 step은 반올림 때문에 원소 수가 예상과 다를 수 있습니다.
- **공식 문서:** [torch.arange](https://docs.pytorch.org/docs/stable/generated/torch.arange.html)

</details>

<details>
<summary><code>model.train(mode=True)</code></summary>

#### `model.train(mode=True)`

- **역할:** 모듈과 하위 모듈을 학습 모드로 전환합니다.
- **입력·반환:** 불리언 모드를 받고 모듈 자신을 반환합니다.
- **이 셀에서 쓰는 이유:** Dropout과 BatchNorm이 학습 시 동작을 사용하도록 합니다.
- **주의점:** gradient를 켜는 함수가 아니므로 `no_grad`와 독립적입니다.
- **공식 문서:** [Module.train](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>optimizer.zero_grad(set_to_none=True)</code></summary>

#### `optimizer.zero_grad(set_to_none=True)`

- **역할:** 파라미터에 누적된 gradient를 비웁니다.
- **입력·반환:** 초기화 방식을 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 현재 mini-batch의 gradient만으로 update하도록 학습 단계를 시작합니다.
- **주의점:** 호출하지 않으면 PyTorch gradient가 기본적으로 누적됩니다.
- **공식 문서:** [Optimizer.zero_grad](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.zero_grad.html)

</details>

<details>
<summary><code>F.cross_entropy(input, target, ...)</code></summary>

#### `F.cross_entropy(input, target, ...)`

- **역할:** logit에 log-softmax와 음의 로그우도를 결합합니다.
- **입력·반환:** 예측·target과 설정을 받고 축약된 loss 또는 같은 shape의 loss를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 objective를 수치화해 역전파 시작점으로 사용합니다.
- **주의점:** 입력 형식, target dtype·shape, `reduction` 의미를 반드시 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\mathcal{L}_{CE}=-\frac{1}{N}\sum_{n=1}^{N}\log\frac{e^{z_{n,y_n}}}{\sum_{c=1}^{C}e^{z_{n,c}}}
$$

- **기호:** $z_{n,c}$는 class logit, $y_n$은 정답 class ID, $C$는 class 수입니다.
- **수식 → 코드:** 함수가 내부에서 `log_softmax`와 정답 class의 음의 로그우도를 결합합니다.
- **직관:** 정답 class 확률이 낮을수록 큰 벌점을 줍니다.
- **shape 확인:** 일반적으로 logits `[N,C,...]`, class-index target `[N,...]`을 사용합니다.
- **공식 문서:** [F.cross_entropy](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html)

</details>

<details>
<summary><code>loss.backward()</code></summary>

#### `loss.backward()`

- **역할:** 스칼라 loss에서 계산 그래프를 거슬러 gradient를 계산합니다.
- **입력·반환:** 스칼라 텐서에서는 보통 인자 없이 호출하며 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** optimizer update에 필요한 각 파라미터의 `.grad`를 채웁니다.
- **주의점:** 같은 그래프를 다시 역전파하려면 별도 보존 설정이 필요합니다.

##### 관련 수식과 코드 연결

$$
\frac{\partial L}{\partial x}=\frac{\partial L}{\partial y}\frac{\partial y}{\partial x},\qquad x.\mathrm{grad}\leftarrow x.\mathrm{grad}+\nabla_xL
$$

- **기호:** $L$은 scalar loss이고 $x$는 계산 그래프의 leaf parameter입니다.
- **수식 → 코드:** `backward()`는 연쇄법칙으로 gradient를 계산해 각 parameter의 `.grad`에 누적합니다.
- **직관:** 파라미터를 직접 바꾸는 호출이 아니라 update에 필요한 미분값을 준비하는 호출입니다.
- **shape 확인:** 각 `.grad`는 대응 parameter와 같은 shape이며 scalar가 아닌 출력에는 seed gradient가 필요합니다.
- **공식 문서:** [Tensor.backward](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.backward.html)

</details>

<details>
<summary><code>optimizer.step(closure=None)</code></summary>

#### `optimizer.step(closure=None)`

- **역할:** 현재 gradient와 optimizer 규칙으로 파라미터를 갱신합니다.
- **입력·반환:** 선택적 closure를 받고 optimizer에 따라 loss 또는 `None`을 반환합니다.
- **이 셀에서 쓰는 이유:** 역전파로 계산한 gradient를 실제 학습 update에 반영합니다.
- **주의점:** 일반적으로 `zero_grad`, forward, `backward` 다음에 호출합니다.

##### 관련 수식과 코드 연결

$$
\theta_{t+1}=\mathcal{U}(\theta_t,\nabla_\theta L_t,s_t;\eta)
$$

- **기호:** $\mathcal U$는 optimizer update, $s_t$는 momentum·moment 상태입니다.
- **수식 → 코드:** `step()`은 이미 `.grad`에 저장된 값을 읽어 선택한 optimizer 규칙을 적용합니다.
- **직관:** gradient 계산과 parameter 갱신을 분리하면 학습 lifecycle을 정확히 추적할 수 있습니다.
- **shape 확인:** parameter shape은 유지되고 값과 optimizer state만 바뀝니다.
- **공식 문서:** [Optimizer.step](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.step.html)

</details>

<details>
<summary><code>tensor.detach()</code></summary>

#### `tensor.detach()`

- **역할:** 같은 저장공간을 공유하면서 현재 계산 그래프에서 분리한 텐서를 만듭니다.
- **입력·반환:** 텐서를 받아 gradient를 추적하지 않는 텐서 view를 반환합니다.
- **이 셀에서 쓰는 이유:** 로그·teacher target·NumPy 변환이 역전파 경로를 만들지 않게 합니다.
- **주의점:** 원본과 저장공간을 공유하므로 in-place 변경은 양쪽에 영향을 줄 수 있습니다.
- **공식 문서:** [Tensor.detach](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.detach.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-2, weight_decay=1e-4)
targets = torch.arange(len(QUERY_PAIRS), device=DEVICE)
losses = []
q_ids, q_mask, d_ids, d_mask = ACCELERATOR.move(
    query_ids, query_mask, document_ids, document_mask
)
model.train()
for step in range(160):
    optimizer.zero_grad(set_to_none=True)
    q_vectors = model.encode(q_ids, q_mask, "query")
    d_vectors = model.encode(d_ids, d_mask, "document")
    logits = q_vectors @ d_vectors.T / 0.08
    loss = (
        torch.nn.functional.cross_entropy(logits, targets)
        + torch.nn.functional.cross_entropy(logits.T, targets)
    ) / 2
    loss.backward()
    optimizer.step()
    losses.append(float(loss.detach()))
assert losses[-1] < losses[0]
print("first/final loss=", round(losses[0], 4), round(losses[-1], 4))

## 6) L2 normalization과 cosine similarity

모든 vector를 unit length로 만들면 cosine similarity는 단순 내적입니다. query와 index에서 서로 다른 정규화 규칙을 사용하면
ranking이 조용히 망가집니다. vector shape와 norm을 계약으로 검증하세요.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.inference_mode,Module.eval,Tensor.cpu,np.linalg.norm,np.allclose -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 5개 · 수식 2개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.inference_mode(mode=True)</code></summary>

#### `torch.inference_mode(mode=True)`

- **역할:** 추론 전용으로 autograd 관련 추적을 더 강하게 비활성화합니다.
- **입력·반환:** 불리언 모드를 받고 context manager 또는 decorator로 동작합니다.
- **이 셀에서 쓰는 이유:** 순수 평가 구간의 오버헤드를 줄입니다.
- **주의점:** 이 모드에서 만든 텐서는 이후 gradient 계산에 쓰기 어려울 수 있습니다.
- **공식 문서:** [torch.inference_mode](https://docs.pytorch.org/docs/stable/generated/torch.inference_mode.html)

</details>

<details>
<summary><code>model.eval()</code></summary>

#### `model.eval()`

- **역할:** 모듈과 하위 모듈을 평가 모드로 전환합니다.
- **입력·반환:** 인자 없이 호출하고 모듈 자신을 반환합니다.
- **이 셀에서 쓰는 이유:** Dropout을 끄고 BatchNorm의 저장 통계를 사용하게 합니다.
- **주의점:** gradient 기록은 끄지 않으므로 필요하면 `no_grad`도 사용합니다.
- **공식 문서:** [Module.eval](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>tensor.cpu()</code></summary>

#### `tensor.cpu()`

- **역할:** 텐서를 CPU 메모리로 옮깁니다.
- **입력·반환:** 텐서를 받아 CPU 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** NumPy 변환이나 CPU 기반 평가 도구에 값을 전달합니다.
- **주의점:** GPU 텐서 이동은 동기화·복사 비용을 만들 수 있습니다.
- **공식 문서:** [Tensor.cpu](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.cpu.html)

</details>

<details>
<summary><code>np.linalg.norm(x, ord=None, axis=None, ...)</code></summary>

#### `np.linalg.norm(x, ord=None, axis=None, ...)`

- **역할:** 벡터나 행렬의 norm을 계산합니다.
- **입력·반환:** 배열과 축·dtype 등의 설정을 받고 배열 또는 수치 결과를 반환합니다.
- **이 셀에서 쓰는 이유:** 데이터 shape과 수치 연산을 논문 실험에 맞게 구성·검증합니다.
- **주의점:** 축, dtype, 복사·view 여부와 빈 배열에서의 동작을 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\lVert x\rVert_p=\left(\sum_i\lvert x_i\rvert^p\right)^{1/p},\qquad \lVert x\rVert_2=\sqrt{\sum_i x_i^2}
$$

- **기호:** $p$는 norm 차수이며 $p=2$가 유클리드 길이입니다.
- **수식 → 코드:** `ord`와 `dim`·`axis`가 어떤 원소 묶음의 크기를 계산할지 정합니다.
- **직관:** 여러 성분을 하나의 거리·크기 값으로 요약합니다.
- **shape 확인:** 지정한 축은 결과에서 제거되며 `keepdim=True`이면 크기 1로 남습니다.
- **공식 문서:** [np.linalg.norm](https://numpy.org/doc/stable/reference/generated/numpy.linalg.norm.html)

</details>

<details>
<summary><code>np.allclose(a, b, rtol=1e-5, atol=1e-8, equal_nan=False)</code></summary>

#### `np.allclose(a, b, rtol=1e-5, atol=1e-8, equal_nan=False)`

- **역할:** 두 배열의 모든 원소가 허용 오차 안에서 가까운지 검사합니다.
- **입력·반환:** 배열 `a`, 기준 배열 `b`, 오차 설정을 받고 Python `bool` 하나를 반환합니다.
- **이 셀에서 쓰는 이유:** 부동소수점 계산 결과를 정확한 값이 아니라 수치적으로 같은지 확인합니다.
- **주의점:** 조건은 `abs(a-b) <= atol + rtol*abs(b)`입니다. `==`와 달리 오차를 허용하고, `b`가 기준이라 비대칭일 수 있으며 1보다 매우 작은 값에는 기본 `atol`이 너무 클 수 있습니다.

##### 관련 수식과 코드 연결

$$
\lvert a_i-b_i\rvert \le \mathrm{atol}+\mathrm{rtol}\,\lvert b_i\rvert
$$

- **기호:** $a_i$는 계산값, $b_i$는 기준값, `atol`은 절대 오차, `rtol`은 상대 오차 허용치입니다.
- **수식 → 코드:** `isclose`는 이 부등식을 원소마다 계산하고, `allclose`는 모든 결과에 논리 AND를 적용합니다.
- **직관:** 완전히 같은 비트인지가 아니라 실험 목적에 충분히 가까운지를 묻는 검사입니다.
- **shape 확인:** 두 입력은 broadcasting 가능해야 합니다. `isclose`는 broadcast shape, `allclose`는 scalar `bool`을 반환합니다.
- **공식 문서:** [np.allclose](https://numpy.org/doc/stable/reference/generated/numpy.allclose.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
@torch.inference_mode()
def dense_encode(texts: list[str], tower: str) -> np.ndarray:
    ids, mask = batch_char_ids(texts)
    model.eval()
    ids, mask = ACCELERATOR.move(ids, mask)
    vectors = model.encode(ids, mask, tower)
    return vectors.cpu().numpy().astype(np.float32)


chunk_vectors = dense_encode([c.searchable_text for c in chunks], "document")
norms = np.linalg.norm(chunk_vectors, axis=1)
assert chunk_vectors.shape == (len(chunks), 48)
assert np.allclose(norms, 1.0, atol=1e-5)
print("vectors=", chunk_vectors.shape, "norm range=", (norms.min(), norms.max()))

## 7) NumPy brute-force vector index

이 데이터에서는 모든 vector와 query의 내적을 계산해도 충분히 빠릅니다. index는 vector와 metadata 길이, dimension,
고유 chunk id를 검증하고 top-k에서 row index를 metadata로 되돌립니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=np.asarray,json.dumps,np.load -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 3개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>np.asarray(a, dtype=None, order=None)</code></summary>

#### `np.asarray(a, dtype=None, order=None)`

- **역할:** 가능하면 복사 없이 입력을 NumPy 배열로 해석합니다.
- **입력·반환:** 배열과 축·dtype 등의 설정을 받고 배열 또는 수치 결과를 반환합니다.
- **이 셀에서 쓰는 이유:** 데이터 shape과 수치 연산을 논문 실험에 맞게 구성·검증합니다.
- **주의점:** 축, dtype, 복사·view 여부와 빈 배열에서의 동작을 확인해야 합니다.
- **공식 문서:** [np.asarray](https://numpy.org/doc/stable/reference/generated/numpy.asarray.html)

</details>

<details>
<summary><code>json.dumps(obj, ensure_ascii=True, indent=None, ...)</code></summary>

#### `json.dumps(obj, ensure_ascii=True, indent=None, ...)`

- **역할:** Python 객체를 JSON 문자열로 직렬화합니다.
- **입력·반환:** 객체를 받고 JSON 문자열을 반환합니다.
- **이 셀에서 쓰는 이유:** metric이나 sample을 파일·로그에 넣기 전 안정된 문자열로 만듭니다.
- **주의점:** 한글 보존에는 `ensure_ascii=False`가 필요하고 객체 종류가 제한됩니다.
- **공식 문서:** [json.dumps](https://docs.python.org/3/library/json.html#json.dumps)

</details>

<details>
<summary><code>np.load(file, allow_pickle=False, ...)</code></summary>

#### `np.load(file, allow_pickle=False, ...)`

- **역할:** NumPy 배열이나 archive를 파일에서 읽습니다.
- **입력·반환:** 배열과 축·dtype 등의 설정을 받고 배열 또는 수치 결과를 반환합니다.
- **이 셀에서 쓰는 이유:** 데이터 shape과 수치 연산을 논문 실험에 맞게 구성·검증합니다.
- **주의점:** 축, dtype, 복사·view 여부와 빈 배열에서의 동작을 확인해야 합니다.
- **공식 문서:** [np.load](https://numpy.org/doc/stable/reference/generated/numpy.load.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
class NumpyVectorIndex:
    def __init__(self, vectors: np.ndarray, metadata: list[dict]):
        vectors = np.asarray(vectors, dtype=np.float32)
        if vectors.ndim != 2 or len(vectors) != len(metadata):
            raise ValueError("vectors must be [N,D] and align with metadata")
        norms = np.linalg.norm(vectors, axis=1, keepdims=True)
        if np.any(norms == 0):
            raise ValueError("zero vector cannot be indexed")
        ids = [row["chunk_id"] for row in metadata]
        if len(ids) != len(set(ids)):
            raise ValueError("chunk_id must be unique")
        self.vectors = vectors / norms
        self.metadata = list(metadata)

    def search(self, query_vector: np.ndarray, top_k: int = 3) -> list[dict]:
        query_vector = np.asarray(query_vector, dtype=np.float32).reshape(-1)
        if query_vector.shape[0] != self.vectors.shape[1] or top_k <= 0:
            raise ValueError("query dimension mismatch or invalid top_k")
        query_vector = query_vector / max(float(np.linalg.norm(query_vector)), 1e-12)
        scores = self.vectors @ query_vector
        order = np.argsort(scores)[::-1][: min(top_k, len(scores))]
        return [
            {**self.metadata[i], "rank": rank, "score": float(scores[i])}
            for rank, i in enumerate(order, start=1)
        ]

    def save(self, folder: Path) -> None:
        folder.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(folder / "vectors.npz", vectors=self.vectors)
        (folder / "metadata.json").write_text(
            json.dumps(self.metadata, ensure_ascii=False, indent=2), encoding="utf-8"
        )

    @classmethod
    def load(cls, folder: Path):
        vectors = np.load(folder / "vectors.npz", allow_pickle=False)["vectors"]
        metadata = json.loads((folder / "metadata.json").read_text(encoding="utf-8"))
        return cls(vectors, metadata)

## 8) index 저장·로드와 id mapping 회귀 테스트

실제 서비스에서는 offline indexer가 만든 artifact를 online retriever가 읽습니다. 여기서는 임시 폴더에 저장해 원본 학습 폴더를
더럽히지 않습니다. 재로드 전후 점수·id가 같아야 합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=Path -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>Path(*pathsegments)</code></summary>

#### `Path(*pathsegments)`

- **역할:** 운영체제에 맞는 파일 경로 객체를 만듭니다.
- **입력·반환:** 경로 조각을 받고 `Path` 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** Windows에서도 문자열 결합 없이 데이터·결과 경로를 구성합니다.
- **주의점:** 상대 경로는 현재 작업 디렉터리를 기준으로 해석됩니다.
- **공식 문서:** [Path](https://docs.python.org/3/library/pathlib.html#pathlib.Path)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
metadata = [asdict(chunk) for chunk in chunks]
dense_index = NumpyVectorIndex(chunk_vectors, metadata)
query_vector = dense_encode(["429 오류의 재시도 방법"], "query")[0]
before = dense_index.search(query_vector, top_k=3)

with tempfile.TemporaryDirectory() as temporary_directory:
    index_path = Path(temporary_directory) / "dense_index"
    dense_index.save(index_path)
    restored_index = NumpyVectorIndex.load(index_path)
    after = restored_index.search(query_vector, top_k=3)
assert [r["chunk_id"] for r in before] == [r["chunk_id"] for r in after]
assert np.allclose([r["score"] for r in before], [r["score"] for r in after])
display(pd.DataFrame(after)[["rank", "source_id", "chunk_id", "score", "title"]])

## 9) retrieval 평가: Recall@k와 MRR

chunk 검색 결과를 원문 `source_id` 기준으로 평가합니다. Recall@k는 정답 문서가 top-k 안에 있는 비율이고,
MRR은 첫 정답 순위의 역수 평균이라 상위 순위를 더 강하게 보상합니다. generation 평가와 분리해야 실패 지점을 찾을 수 있습니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=np.mean -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>np.mean(a, axis=None, dtype=None, ...)</code></summary>

#### `np.mean(a, axis=None, dtype=None, ...)`

- **역할:** 지정 축의 산술평균을 계산합니다.
- **입력·반환:** 배열과 축·dtype 등의 설정을 받고 배열 또는 수치 결과를 반환합니다.
- **이 셀에서 쓰는 이유:** 데이터 shape과 수치 연산을 논문 실험에 맞게 구성·검증합니다.
- **주의점:** 축, dtype, 복사·view 여부와 빈 배열에서의 동작을 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\bar x=\frac{1}{N}\sum_{i=1}^{N}x_i
$$

- **기호:** $N$은 선택한 축의 원소 수입니다.
- **수식 → 코드:** `dim` 또는 `axis`가 평균에 포함할 원소와 결과에서 사라질 축을 결정합니다.
- **직관:** 여러 관측값을 중심 위치 하나로 요약합니다.
- **shape 확인:** `keepdim=False`이면 축약한 축이 제거되고 참이면 크기 1로 남습니다.
- **공식 문서:** [np.mean](https://numpy.org/doc/stable/reference/generated/numpy.mean.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
EVAL_CASES = QUERY_PAIRS + [
    ("인증 링크가 이메일로 오지 않아요", "kb-001"),
    ("택배 위치가 48시간 동안 그대로예요", "kb-006"),
    ("Retry-After와 지수 백오프가 궁금해요", "kb-010"),
]


def evaluate(search_fn, cases, top_k: int = 3) -> dict:
    hits, reciprocal_ranks = 0, []
    for eval_query, expected_id in cases:
        results = search_fn(eval_query, top_k)
        source_ids = [row["source_id"] for row in results]
        hits += expected_id in source_ids
        reciprocal_ranks.append(
            1 / (source_ids.index(expected_id) + 1)
            if expected_id in source_ids
            else 0.0
        )
    return {"recall_at_k": hits / len(cases), "mrr": float(np.mean(reciprocal_ranks))}

In [ ]:
def dense_search(text: str, top_k: int = 3) -> list[dict]:
    return dense_index.search(dense_encode([text], "query")[0], top_k=top_k)


comparison = pd.DataFrame(
    {"retriever": name, "k": k, **evaluate(search_fn, EVAL_CASES, k)}
    for name, search_fn in (
        ("sparse-char-tfidf", sparse_search),
        ("tiny-dense", dense_search),
    )
    for k in (1, 3, 5)
)
assert comparison["recall_at_k"].between(0, 1).all()
display(comparison.round(3))

## 10) 결과를 해석하는 법

작은 dense 모델은 훈련 query를 잘 외우지만 새로운 표현에는 약할 수
있습니다. TF-IDF는 정확한 오류 코드와 단어가 강하고, dense model은
표현이 달라도 가까운 의미를 목표로 합니다. 현업에서는 두 score를
calibration하거나 reciprocal-rank fusion으로 합치고,
카테고리/권한/버전 metadata filter를 적용합니다. 모델의 train metric보다 **고정된 미관측 retrieval 평가셋**을 우선하세요.

## 선택 확장: FAISS

데이터가 수십만 개를 넘으면 `IndexFlatIP`(정확 검색)나 HNSW/IVF(근사 검색)를 고려할 수 있습니다. FAISS를 쓸 때도 입력을
`float32`와 L2 unit vector로 맞추고, FAISS row id ↔ metadata 저장소 id 매핑을 별도로 보존하세요.
**이 실습은 FAISS가 없어도 완전히 동작**하며 NumPy 결과가 정확 검색 기준선입니다.

## 직접 해볼 실험

1. `chunk_size/overlap`을 바꾸고 chunk 수, Recall@3, context 중복을 기록하세요.
2. dense output dimension을 16/48/128로 바꾸고 속도와 MRR을 비교하세요.
3. 평가 query를 직접 20개 추가하되 train query와 분리하세요.
4. sparse와 dense 순위를 RRF로 합치고 개선된 query와 악화된 query를 각각 설명하세요.
5. metadata에서 `category` filter를 넣되 filter 전후에 top-k 정의가 어떻게 달라지는지 검증하세요.

완료 기준: embedding 값 자체보다 **데이터 계약 → row/id 매핑 → 평가 → 저장/로드 일관성**을 설명할 수 있습니다.